In [1]:
# convert_real_to_white_theme.py
import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# ================= 配置 =================
# 真实数据路径
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "QAM64",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "QAM16",
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK", # 修正命名
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/OQPSK-03-24-25-10h54m02s6.iq": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

# 输出目录
OUTPUT_DIR = "./dataset_real_white_theme"
SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 核心计算 (保持 RAD 算法一致) =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_rad_features(iq_complex_data, img_size=64):
    """
    计算 R, A, D 三个通道的数值 (0.0 ~ 1.0)
    """
    if iq_complex_data.dim() == 1: iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    if iq_data.numel() < 100: return torch.zeros(3, img_size, img_size)

    # 归一化
    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    # 1. R 通道
    hist_r = fast_hist2d(norm_iq.real, norm_iq.imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    # 2. A 通道
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    hist_a = fast_hist2d(phase, mag, bins=img_size, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    
    # 3. D 通道
    diff = norm_iq[1:] * torch.conj(norm_iq[:-1])
    d_scale = torch.sqrt(torch.mean(diff.real**2 + diff.imag**2)) + 1e-9
    diff = diff / (3.0 * d_scale)
    hist_d = fast_hist2d(diff.real, diff.imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    
    def process(h):
        h = torch.log1p(h)
        return (h - h.min()) / (h.max() - h.min() + 1e-8)
    
    # 返回原始的 0-1 浮点特征，不在这里做颜色映射
    img_tensor = torch.stack([process(hist_r), process(hist_a), process(hist_d)], dim=0) 
    return torch.flip(img_tensor, [1]) 

def save_as_white_theme(tensor, path):
    """
    [关键修改] 保存为白底风格 (CMY 减色模式)
    输入 tensor: (3, H, W) 范围 0~1，代表特征强度
    """
    tensor = tensor.detach().cpu() # (3, H, W)
    
    # 逻辑反转：特征越强，像素越暗 (1 - value)
    # 这相当于把 RGB 变成了 CMY
    # R (Constellation) -> Cyan (青色)
    # A (Polar)         -> Magenta (洋红)
    # D (Diff)          -> Yellow (黄色)
    # 背景 (0,0,0)      -> White (1,1,1)
    
    inverted_tensor = 1.0 - tensor
    
    arr = (inverted_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    Image.fromarray(arr, mode='RGB').save(path)

# ================= 主程序 =================

if __name__ == "__main__":
    print("===== 开始转换真实数据 (白底风格) =====")
    print(f"输出目录: {os.path.abspath(OUTPUT_DIR)}")
    
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skipping missing file: {file_path}")
            continue
            
        print(f"Processing {label}...")
        save_dir = os.path.join(OUTPUT_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        
        try:
            # 读取 IQ
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q).astype(np.complex64) / 32767.0
            
            # 为了演示，每类只生成前 20 张图，方便快速查看
            num_slices = min(20, len(iq_complex) // SAMPLES_PER_IMAGE)
            
            for i in tqdm(range(num_slices)):
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                chunk_tensor = torch.from_numpy(chunk).to(DEVICE)
                
                # 1. 计算特征 (RAD)
                features = generate_rad_features(chunk_tensor, IMAGE_SIZE)
                
                # 2. 保存为白底图
                save_as_white_theme(features, os.path.join(save_dir, f"real_white_{i:04d}.png"))
                
        except Exception as e:
            print(f"Error: {e}")
            
    print("\n转换完成！请打开文件夹查看效果。")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


===== 开始转换真实数据 (白底风格) =====
输出目录: e:\Project\AMR-Benchmark\Other\dataset_real_white_theme
Processing BPSK...


100%|██████████| 20/20 [00:00<00:00, 20.21it/s]


Processing QAM64...


100%|██████████| 20/20 [00:00<00:00, 319.03it/s]


Processing QAM16...


... (10 outputs left out)

100%|██████████| 20/20 [00:00<00:00, 324.69it/s]


Processing PI4DQPSK...


100%|██████████| 20/20 [00:00<00:00, 307.45it/s]


转换完成！请打开文件夹查看效果。
